# 🐍 Python para Análisis de Datos — Guía para Ingenieros Industriales
**Nivel:** Intermedio (conoces las bases de Python)  
**Objetivo:** Dominar `pandas`, `numpy`, `matplotlib` y `seaborn` para analizar datos industriales reales.

---

## ¿Por qué Python para Análisis de Datos?

Como Ingeniero Industrial, tus datos provienen de:
- **Líneas de producción**: tiempos de ciclo, tasas de defectos, OEE
- **Cadena de suministro**: inventarios, tiempos de entrega, costos
- **Mantenimiento**: fallas de equipos, tiempos de reparación (MTTR, MTBF)
- **Calidad**: mediciones de piezas, auditorías, devoluciones de clientes

Python te permite ir desde los datos crudos hasta decisiones accionables en minutos.

> 💡 **Filosofía del curso:** Cada concepto va acompañado de un ejemplo industrial real y un ejercicio práctico.

## 📦 Librerías que usaremos

| Librería | Para qué sirve | Equivalente en Excel |
|----------|---------------|---------------------|
| `pandas` | Manipular tablas de datos | Hojas de cálculo / Power Query |
| `numpy`  | Cálculos numéricos rápidos | Fórmulas matemáticas |
| `matplotlib` | Gráficas básicas | Gráficos de Excel |
| `seaborn` | Gráficas estadísticas elegantes | — |
| `scipy.stats` | Estadística avanzada | Análisis de datos de Excel |

In [ ]:
# Importamos todas las librerías que necesitaremos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración estética de las gráficas
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

print("✅ Todas las librerías importadas correctamente")
print(f"   pandas  versión: {pd.__version__}")
print(f"   numpy   versión: {np.__version__}")

---
## 1️⃣ Creación y exploración de DataFrames

### Contexto real:
Eres el analista de producción de una planta que fabrica componentes electrónicos.  
Tienes el registro de producción de los últimos 30 días: fecha, turno, máquina, 
piezas producidas, piezas defectuosas y tiempo de paro.

In [ ]:
# Creamos un dataset simulando datos reales de producción industrial
np.random.seed(42)
n = 90  # 30 días × 3 turnos

turnos     = ['Mañana', 'Tarde', 'Noche'] * 30
maquinas   = np.random.choice(['Máquina A', 'Máquina B', 'Máquina C'], n)
produccion = np.random.randint(450, 600, n)
defectos   = np.random.randint(2, 25, n)
tiempo_paro_min = np.random.exponential(scale=15, size=n).round(1)

fechas = pd.date_range(start='2024-01-01', periods=30, freq='D')
fechas_rep = np.repeat(fechas, 3)  # 3 turnos por día

df = pd.DataFrame({
    'fecha':            fechas_rep,
    'turno':            turnos,
    'maquina':          maquinas,
    'piezas_producidas': produccion,
    'piezas_defectuosas': defectos,
    'tiempo_paro_min':  tiempo_paro_min
})

# Calculamos columnas derivadas
df['tasa_defectos_%'] = (df['piezas_defectuosas'] / df['piezas_producidas'] * 100).round(2)
df['piezas_buenas']   = df['piezas_producidas'] - df['piezas_defectuosas']

print("Dataset de producción creado:")
print(f"  Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
df.head(6)

In [ ]:
# --- Exploración inicial — SIEMPRE empieza con esto ---
print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print()
print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
df.describe().round(2)

In [ ]:
# Verificar valores nulos — crítico antes de cualquier análisis
print("=== VALORES NULOS POR COLUMNA ===")
print(df.isnull().sum())
print()
# En datasets industriales reales, los nulos suelen ser sensores desconectados
# o turnos sin producción. Hay que manejarlos ANTES de analizar.

---
## 2️⃣ Filtrado y selección de datos

> 💡 **Regla de oro:** Filtra primero, analiza después. No trabajes con más datos de los necesarios.

In [ ]:
# --- Seleccionar columnas específicas ---
resumen_basico = df[['fecha', 'turno', 'maquina', 'piezas_producidas', 'tasa_defectos_%']]
print("Columnas seleccionadas:")
resumen_basico.head(4)

In [ ]:
# --- Filtrar filas con condiciones ---

# ¿Cuándo la tasa de defectos supera el 4%? (umbral de alerta)
alta_defectos = df[df['tasa_defectos_%'] > 4.0]
print(f"Registros con tasa de defectos > 4%: {len(alta_defectos)}")
alta_defectos[['fecha', 'turno', 'maquina', 'tasa_defectos_%']].head(6)

In [ ]:
# Filtros múltiples: turno noche Y más de 15 min de paro
turno_noche_paro = df[(df['turno'] == 'Noche') & (df['tiempo_paro_min'] > 15)]
print(f"Turno noche con paro > 15 min: {len(turno_noche_paro)} registros")
turno_noche_paro[['fecha', 'maquina', 'piezas_producidas', 'tiempo_paro_min']].head(5)

---
## 3️⃣ Agrupación y resúmenes estadísticos (groupby)

Este es el equivalente a las **tablas dinámicas** de Excel, pero mucho más poderoso.

In [ ]:
# --- Producción y defectos por turno ---
resumen_turno = df.groupby('turno').agg(
    total_producido   = ('piezas_producidas',  'sum'),
    total_defectos    = ('piezas_defectuosas', 'sum'),
    tasa_defectos_prom = ('tasa_defectos_%',   'mean'),
    paro_promedio_min  = ('tiempo_paro_min',    'mean')
).round(2)

resumen_turno['tasa_defectos_global_%'] = (
    resumen_turno['total_defectos'] / resumen_turno['total_producido'] * 100
).round(2)

print("=== RESUMEN POR TURNO ===")
resumen_turno

In [ ]:
# --- Comparación por máquina ---
resumen_maquina = df.groupby('maquina').agg(
    piezas_buenas_total = ('piezas_buenas',      'sum'),
    defectos_total      = ('piezas_defectuosas', 'sum'),
    paro_total_min      = ('tiempo_paro_min',    'sum'),
    tasa_defectos_prom  = ('tasa_defectos_%',    'mean')
).round(2)

# OEE simplificado: disponibilidad basada en paros
horas_turno = 8 * 60  # minutos por turno
n_turnos_maq = df.groupby('maquina').size()
resumen_maquina['disponibilidad_%'] = (
    1 - resumen_maquina['paro_total_min'] / (horas_turno * n_turnos_maq) 
).clip(0, 1).mul(100).round(1)

print("=== RESUMEN POR MÁQUINA ===")
resumen_maquina

---
## 4️⃣ Visualización de datos industriales

> 💡 **Regla profesional:** Siempre grafica los datos ANTES de calcular estadísticas.
> Los números pueden engañar; las gráficas revelan patrones ocultos.

In [ ]:
# --- Gráfico de barras: producción por turno ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: producción total
colors = ['#2196F3', '#FF9800', '#4CAF50']
resumen_turno['total_producido'].plot(
    kind='bar', ax=axes[0], color=colors, edgecolor='white', width=0.6
)
axes[0].set_title('Piezas Producidas por Turno', fontweight='bold')
axes[0].set_xlabel('Turno')
axes[0].set_ylabel('Total de Piezas')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(resumen_turno['total_producido']):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# Derecha: tasa de defectos
resumen_turno['tasa_defectos_global_%'].plot(
    kind='bar', ax=axes[1], color=colors, edgecolor='white', width=0.6
)
axes[1].axhline(y=3, color='red', linestyle='--', linewidth=2, label='Límite 3%')
axes[1].set_title('Tasa de Defectos por Turno (%)', fontweight='bold')
axes[1].set_xlabel('Turno')
axes[1].set_ylabel('Tasa de Defectos (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()

plt.tight_layout()
plt.savefig('produccion_por_turno.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Gráfica guardada como 'produccion_por_turno.png'")

In [ ]:
# --- Serie de tiempo: tendencia de defectos diarios ---
defectos_diarios = df.groupby('fecha')['tasa_defectos_%'].mean()

plt.figure(figsize=(12, 5))
plt.plot(defectos_diarios.index, defectos_diarios.values,
         color='#E53935', linewidth=2, marker='o', markersize=4, label='Tasa diaria')

# Línea de tendencia (regresión lineal)
x_num = np.arange(len(defectos_diarios))
slope, intercept, r, p, _ = stats.linregress(x_num, defectos_diarios.values)
tendencia = slope * x_num + intercept
plt.plot(defectos_diarios.index, tendencia,
         color='navy', linewidth=2, linestyle='--',
         label=f'Tendencia (pendiente={slope:.3f}%/día)')

plt.axhline(y=3.0, color='orange', linestyle=':', linewidth=2, label='Alerta: 3%')
plt.fill_between(defectos_diarios.index, defectos_diarios.values, alpha=0.1, color='red')
plt.title('Tendencia de Tasa de Defectos — Enero 2024', fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Tasa de Defectos (%)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('tendencia_defectos.png', dpi=120, bbox_inches='tight')
plt.show()

if slope > 0:
    print(f"⚠️  ALERTA: La tasa de defectos muestra una tendencia CRECIENTE de {slope:.4f}%/día")
else:
    print(f"✅ La tasa de defectos muestra una tendencia decreciente de {slope:.4f}%/día")

In [ ]:
# --- Boxplot por máquina: comparar variabilidad ---
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df,
    x='maquina', y='tasa_defectos_%',
    palette=['#42A5F5', '#EF5350', '#66BB6A'],
    width=0.5
)
sns.stripplot(
    data=df,
    x='maquina', y='tasa_defectos_%',
    color='black', alpha=0.3, size=4
)
plt.axhline(y=3.0, color='red', linestyle='--', linewidth=2, label='Umbral 3%')
plt.title('Distribución de Tasa de Defectos por Máquina', fontweight='bold')
plt.xlabel('Máquina')
plt.ylabel('Tasa de Defectos (%)')
plt.legend()
plt.tight_layout()
plt.savefig('boxplot_maquinas.png', dpi=120, bbox_inches='tight')
plt.show()
print("💡 Los puntos muestran cada observación individual (transparentes para ver densidad)")

In [ ]:
# --- Heatmap: correlación entre variables ---
variables_num = df[['piezas_producidas', 'piezas_defectuosas',
                     'tiempo_paro_min', 'tasa_defectos_%']]
correlacion = variables_num.corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(correlacion, dtype=bool))  # solo mitad inferior
sns.heatmap(
    correlacion,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    mask=mask,
    square=True,
    linewidths=1,
    cbar_kws={'label': 'Coeficiente de correlación'}
)
plt.title('Mapa de Correlaciones — Variables de Producción', fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_correlaciones.png', dpi=120, bbox_inches='tight')
plt.show()
print("💡 Valores cercanos a +1 = correlación positiva fuerte, -1 = negativa fuerte, 0 = sin correlación")

---
## 5️⃣ Estadística aplicada al proceso

### Test de hipótesis: ¿Hay diferencia real entre turnos?

Cuando comparamos turnos, la pregunta real es:  
*"¿La diferencia que veo es real o podría ser solo azar (variación aleatoria)?"*  
Para responder esto usamos **pruebas estadísticas**.

In [ ]:
# --- Comparar tasa de defectos entre turnos con ANOVA ---
turno_man = df[df['turno'] == 'Mañana']['tasa_defectos_%']
turno_tar = df[df['turno'] == 'Tarde']['tasa_defectos_%']
turno_noc = df[df['turno'] == 'Noche']['tasa_defectos_%']

f_stat, p_value = stats.f_oneway(turno_man, turno_tar, turno_noc)

print("=== ANOVA: Diferencia de defectos entre turnos ===")
print(f"Estadístico F:  {f_stat:.4f}")
print(f"Valor p:        {p_value:.4f}")
print()
if p_value < 0.05:
    print("✅ CONCLUSIÓN: Existe diferencia ESTADÍSTICAMENTE SIGNIFICATIVA entre turnos (p < 0.05)")
    print("   → Hay que investigar qué causa la diferencia: operadores, fatiga, condiciones.")
else:
    print("ℹ️  CONCLUSIÓN: No hay diferencia significativa entre turnos (p ≥ 0.05)")
    print("   → La variación observada podría ser aleatoria.")

print()
print("Media de defectos por turno:")
for nombre, datos in [('Mañana', turno_man), ('Tarde', turno_tar), ('Noche', turno_noc)]:
    print(f"  {nombre}: {datos.mean():.2f}% (±{datos.std():.2f}%)")

---
## 🏋️ Ejercicios para practicar

Usando el DataFrame `df` que creamos, resuelve los siguientes ejercicios:

### Ejercicio 1
Calcula el **OEE simplificado** por máquina usando solo la disponibilidad:  
`Disponibilidad = 1 - (tiempo_paro / tiempo_total_disponible)`  
Considera 8 horas por turno y 30 turnos por máquina.

### Ejercicio 2
Identifica los **5 días con mayor cantidad de defectos absolutos** (suma de las 3 máquinas). Grafícalos en un gráfico de barras.

### Ejercicio 3
Crea un nuevo DataFrame que muestre, para cada **máquina + turno**, el promedio de `tasa_defectos_%`. ¿Qué combinación es la más problemática?

### Ejercicio 4
Realiza un **test t de Student** para comparar el tiempo de paro promedio entre la Máquina A y la Máquina B. ¿Hay diferencia significativa? (Usa `scipy.stats.ttest_ind`)

In [ ]:
# === SOLUCIÓN EJERCICIO 1: OEE simplificado ===
tiempo_total_min = 8 * 60 * 30  # 8h × 60min × 30 turnos por máquina

oee_df = df.groupby('maquina').agg(
    paro_total=('tiempo_paro_min', 'sum')
)
oee_df['disponibilidad_%'] = ((1 - oee_df['paro_total'] / tiempo_total_min) * 100).round(1)
oee_df['estado'] = oee_df['disponibilidad_%'].apply(
    lambda x: '🟢 Bueno' if x >= 90 else ('🟡 Aceptable' if x >= 80 else '🔴 Crítico')
)
print("=== OEE SIMPLIFICADO POR MÁQUINA ===")
print(oee_df)

In [ ]:
# === SOLUCIÓN EJERCICIO 3: Cruce máquina × turno ===
cruce = df.groupby(['maquina', 'turno'])['tasa_defectos_%'].mean().round(2).unstack()
print("=== TASA DE DEFECTOS PROMEDIO: MÁQUINA × TURNO ===")
print(cruce)
print()
# Encontrar la peor combinación
idx_max = df.groupby(['maquina', 'turno'])['tasa_defectos_%'].mean().idxmax()
print(f"⚠️  Combinación más problemática: {idx_max[0]} — Turno {idx_max[1]}")